# 15b · eval — **acm** / transfer · seed 2,3

1모델 × 2 seed × 5 rep = **10 run** (이 창 몫).

**150k 체크포인트 × 5 rep × 500 ep** (rep 마다 env seed 변경). 끝난 run 은 자동 skip.


## ⚠️ 이 노트북은 **B 몫**(seed 2,3)만 돌린다 — GPU 2장짜리 노드용

| 노트북 | seed | GPU |
|---|---|---|
| **15b (이 창)** | **2, 3** | 이 노드의 GPU **0, 1** (자동) |
| 15a_eval_acm_transfer_seed01.ipynb | 0, 1 | 그 노드의 GPU 0, 1 |

**GPU 는 노드마다 0번부터 자동 배정**된다 (`cf.part('B')` → 이 노드에 보이는 GPU 를 seed 수만큼).
2-GPU 노드가 두 대면 A 노드에서 `a` 를, B 노드에서 `b` 를 돌리면 된다.

> ⚠️ 없는 GPU 를 지정하면 학습이 `RuntimeError: 0 active drivers` 로 죽는다(장치가 안 보임).
> **한 노드(4-GPU)에서 a·b 를 동시에** 띄울 때만 서로 밟지 않게 직접 나눌 것:
> `SEEDS, GPUS = cf.part('B', gpus=[2, 3])`

둘 다 끝나면 seed 4개가 모여 리포트(`09`/`10`/`18`)에서 **자동으로 합쳐진다**.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK        = cf.SHORT_SIM       # 'transfer'
SEEDS, GPUS = cf.part('B')   # seed [2,3] + 이 노드의 GPU 0,1 (자동)
TAGS        = cf.GROUP_ACM
# 한 4-GPU 노드에서 a·b 동시 실행 시:  SEEDS, GPUS = cf.part('B', gpus=[2, 3])
REPS = list(range(cf.EVAL_REPEATS))
N_EP = cf.EVAL_N_EP

print('task:', TASK, '| ckpt', f'{cf.CKPT_STEP:,}', '| reps', REPS, '| n_ep', N_EP)
print('seeds:', SEEDS, '| GPU:', GPUS)
print('eval:', TAGS, '| 이 창의 run:', len(TAGS) * len(SEEDS) * len(REPS))

## 사전 확인 — 150k 체크포인트 (transfer)

In [ ]:
ok = cf.print_ckpt_status(TAGS, SEEDS, TASK)

## 반복 eval

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, gpus=GPUS, n_episodes=N_EP)

## 결과 (이 창의 seed 만) — 전체 표는 두 창이 다 끝난 뒤 `09_report_sr` / `18_report_horizon`

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP)